# NB01: Data Collection

This notebook collects finished match results for the 2025-26 season of Europe's "big five" leagues from the [football-data.org](https://www.football-data.org/) free API, to investigate whether home advantage is equally strong across leagues.

In [1]:
import json
import time
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("FOOTBALL_DATA_API_KEY")
BASE_URL = "https://api.football-data.org/v4"
HEADERS = {"X-Auth-Token": API_KEY}

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Choosing my leagues and season

I'm comparing the Premier League, La Liga, Bundesliga, Serie A and Ligue 1 - the five leagues most commonly grouped as Europe's strongest domestic competitions. Restricting to one completed season (2025-26) means every league has a full, fixed set of matchdays and no in-progress fixtures that would bias the home/away split.

football-data.org's free tier gives access to all five competitions and labels a season by its starting year, so `season=2025` here means the 2025-26 season, which finished in May 2026.

In [2]:
# The "big five" European leagues, identified by their football-data.org competition codes
COMPETITIONS = {
    "PL": "Premier League (England)",
    "PD": "La Liga (Spain)",
    "BL1": "Bundesliga (Germany)",
    "SA": "Serie A (Italy)",
    "FL1": "Ligue 1 (France)",
}

SEASON = 2025

## Fetching finished matches

For each competition I pull only `status=FINISHED` matches for the 2025-26 season and save the raw API response as-is under `data/raw/`, one JSON file per league. This keeps the original source data untouched, before any cleaning happens in NB02.

The free plan allows 10 requests per minute; five sequential calls comfortably fit, but I still add a short pause between requests to stay well under the limit.

In [3]:
for code, name in COMPETITIONS.items():
    url = f"{BASE_URL}/competitions/{code}/matches"
    params = {"season": SEASON, "status": "FINISHED"}

    response = requests.get(url, headers=HEADERS, params=params)
    response.raise_for_status()
    payload = response.json()

    out_path = RAW_DIR / f"{code}_matches_{SEASON}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(f"{name} ({code}): {len(payload['matches'])} finished matches saved to {out_path}")
    time.sleep(6)

Premier League (England) (PL): 380 finished matches saved to data\raw\PL_matches_2025.json


La Liga (Spain) (PD): 380 finished matches saved to data\raw\PD_matches_2025.json


Bundesliga (Germany) (BL1): 306 finished matches saved to data\raw\BL1_matches_2025.json


Serie A (Italy) (SA): 380 finished matches saved to data\raw\SA_matches_2025.json


Ligue 1 (France) (FL1): 305 finished matches saved to data\raw\FL1_matches_2025.json


## Checking what we collected

Quick sanity check: reload each saved file and confirm the match count and date range look right (a full season, not a partial one).

In [4]:
for code, name in COMPETITIONS.items():
    path = RAW_DIR / f"{code}_matches_{SEASON}.json"
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)

    matches = payload["matches"]
    dates = sorted(m["utcDate"] for m in matches)
    print(f"{name}: {len(matches)} matches, {dates[0][:10]} to {dates[-1][:10]}")

Premier League (England): 380 matches, 2025-08-15 to 2026-05-24
La Liga (Spain): 380 matches, 2025-08-15 to 2026-05-24
Bundesliga (Germany): 306 matches, 2025-08-22 to 2026-05-16
Serie A (Italy): 380 matches, 2025-08-23 to 2026-05-24
Ligue 1 (France): 305 matches, 2025-08-15 to 2026-05-17
